# Comprehensive Model Evaluation: XGBoost Baseline vs PyTorch MLP
This notebook trains both the baseline tree model (XGBoost) and our deep learning model (FraudMLP) on the IEEE-CIS Fraud Detection dataset.
We utilize the standardized evaluation metrics developed in `src.evaluation.metrics` to compare their capabilities side-by-side.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.append(os.path.abspath('..'))

from src.data_prep.data_loader import prepare_data
from src.features.build_features import build_features, get_feature_pipeline
from src.models.tree_models import get_xgboost_model
from src.models.pytorch_mlp import FraudMLP
from src.evaluation.metrics import print_evaluation_report

# PyTorch
import torch
from torch.utils.data import DataLoader, TensorDataset
from src.models.train_nn import train_nn

# Ignore warnings for clean display
import warnings
warnings.filterwarnings('ignore')

## 1. Data Ingestion & Feature Engineering
We load the dataset using our memory-efficient loader, then generate explicit features like Magic UIDs and D-column aggregations.

In [ ]:
# NOTE: Ensure you have unzipped the Kaggle data into `data/raw/`
# Since data paths depend on your local unzipping, we use a mock synthetic dataset here for demonstration if raw files are missing
TRAIN_TRANS = "../data/raw/train_transaction.csv"
TRAIN_ID = "../data/raw/train_identity.csv"

try:
    print("Loading actual data...")
    X, y = prepare_data(TRAIN_TRANS, TRAIN_ID)
    print("Building features...")
    X = build_features(X)
except FileNotFoundError:
    print("Raw Kaggle data not found in data/raw/. Generating synthetics for demonstration of pipeline mechanics...")
    np.random.seed(42)
    n_samples = 10000
    X = pd.DataFrame(np.random.rand(n_samples, 20), columns=[f'feature_{i}' for i in range(20)])
    X['TransactionDT'] = np.random.randint(0, 500000, size=n_samples)
    y = pd.Series(np.random.choice([0, 1], size=n_samples, p=[0.965, 0.035]), name='isFraud')

# Standard Train/Test Temporal Split (Mocked for 80/20 if no DT logic is strict)
train_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:train_idx], X.iloc[train_idx:]
y_train, y_test = y.iloc[:train_idx], y.iloc[train_idx:]

print(f"Train shapes: {X_train.shape}, {y_train.shape}")
print(f"Test shapes: {X_test.shape}, {y_test.shape}")

## 2. Preprocessing & Encoding
Transform datasets using our custom pipelining strategies designed for robust Categorical Frequency mappings.

In [ ]:
# Prepare generic preprocessor pipeline
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

pipeline = get_feature_pipeline(num_cols, cat_cols)
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

## 3. Training & Evaluating Baseline (XGBoost)

In [ ]:
xgb_model = get_xgboost_model()
print("Training Baseline XGBoost...")
xgb_model.fit(X_train_processed, y_train)

print("Inferencing...")
xgb_preds = xgb_model.predict_proba(X_test_processed)[:, 1]

print("\nRESULTS: Baseline XGBoost")
print_evaluation_report(y_test, xgb_preds)

## 4. Training & Evaluating Deep Learning (FraudMLP)
We feed the identically processed data into the PyTorch Neural Network, leveraging Focal Loss and Early Stopping.

In [ ]:
# Convert arrays to explicit PyTorch float32 Tensors
X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# PyTorch DataLoaders
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=1024, shuffle=True)
val_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=1024, shuffle=False)

input_dim = X_train_tensor.shape[1]
mlp_model = FraudMLP(input_dim=input_dim, hidden_dim=256, dropout_rate=0.3)

print("Training Deep Learning Core (FraudMLP)...")
# Train for small epochs as demonstration
mlp_model = train_nn(mlp_model, train_loader, val_loader, epochs=5, lr=0.001)

print("\nInferencing PyTorch...")
mlp_model.eval()
with torch.no_grad():
    mlp_preds = torch.sigmoid(mlp_model(X_test_tensor)).squeeze().numpy()

print("\nRESULTS: Deep Learning (FraudMLP)")
print_evaluation_report(y_test, mlp_preds)

## 5. Performance Comparison Visualization
We leverage precision-recall tradeoff visualizations.

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_xgb, recall_xgb, _ = precision_recall_curve(y_test, xgb_preds)
precision_mlp, recall_mlp, _ = precision_recall_curve(y_test, mlp_preds)

plt.figure(figsize=(10, 6))
plt.plot(recall_xgb, precision_xgb, label='XGBoost Baseline', color='blue', lw=2)
plt.plot(recall_mlp, precision_mlp, label='PyTorch MLP', color='red', lw=2, linestyle='--')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve: XGBoost vs FraudMLP on IEEE-CIS')
plt.legend()
plt.grid(True)
plt.show()